# PyTorch 快速上手

这个 notebook 用来做「边看边试」的练习。改数字、按 `Shift+Enter` 重跑，比读文档快得多。

**第一次用之前**：右上角内核选 `Python 3 (ipykernel)`，也就是本项目 `.venv` 里的那个解释器。

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

print("torch", torch.__version__)
print("CUDA 可用:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("显卡:", torch.cuda.get_device_name(0))

device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 1. 张量的 shape 是唯一重要的东西

养成习惯：每写一步就问自己「现在 shape 是多少」。

In [ ]:
x = torch.randn(4, 3)
print("x          ", x.shape)
print("x.T        ", x.T.shape)
print("x.sum(0)   ", x.sum(0).shape)   # 沿第 0 维求和
print("x.sum(1)   ", x.sum(1).shape)
print("x @ x.T    ", (x @ x.T).shape)
print("unsqueeze  ", x.unsqueeze(0).shape)
print("flatten    ", x.flatten().shape)

## 2. 自动求导：梯度是累加的

跑两次 `backward()`，看 `x.grad` 怎么变。这是新手最常见的 bug 来源。

In [ ]:
x = torch.tensor(3.0, requires_grad=True)

(x ** 2).backward()
print("第一次 backward:", x.grad.item())

(x ** 2).backward()
print("第二次 backward:", x.grad.item(), " <- 变成 12 了，因为累加")

x.grad = None
(x ** 2).backward()
print("清零后再来:", x.grad.item())

## 3. 训练一个网络，并把拟合曲线画出来

画图是排查训练问题最有效的手段：损失曲线不降、或者降了但震荡，一眼就能看出来。

In [ ]:
import torch.nn as nn

torch.manual_seed(0)
xs = torch.linspace(-3, 3, 500).unsqueeze(1)
ys = torch.sin(xs) + 0.5 * xs + 0.1 * torch.randn(500, 1)

model = nn.Sequential(
    nn.Linear(1, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1),
).to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

xs_d, ys_d = xs.to(device), ys.to(device)
history = []
for epoch in range(400):
    loss = loss_fn(model(xs_d), ys_d)
    opt.zero_grad()
    loss.backward()
    opt.step()
    history.append(loss.item())

print("最终 loss:", history[-1])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

with torch.no_grad():
    pred = model(xs_d).cpu()

axes[0].scatter(xs.numpy(), ys.numpy(), s=8, alpha=0.4, label="真实数据")
axes[0].plot(xs.numpy(), pred.numpy(), color="red", label="模型预测")
axes[0].legend()
axes[0].set_title("拟合效果")

axes[1].plot(history)
axes[1].set_yscale("log")
axes[1].set_title("训练损失曲线")
axes[1].set_xlabel("epoch")

plt.tight_layout()
plt.show()

## 练习

1. 把 `nn.ReLU()` 全换成 `nn.Identity()`，重新训练，看看还能不能拟合曲线（这是理解「非线性」最快的实验）。
2. 把 `lr` 从 `1e-2` 改成 `1.0`，观察损失曲线变成什么样。
3. 把隐藏层宽度从 64 改成 8，看欠拟合长什么样。